# 🎤 RVC v2: Персональный голос для песен (Сергей Трофимов — Аты-баты)

**Инструкция: Нажмите `Среда выполнения` -> `Выполнить все` (Runtime -> Run all).**

Пайплайн выполнится полностью автоматически за **~10–12 минут**:
1. Разворачивается оптимизированная среда RVC v2 (Applio) на GPU Tesla T4.
2. Подтягивается ваш чистый 9-минутный голос (`Testr2.wav`) и трек `Сергей Трофимов - Аты-баты`.
3. Извлекаются частоты связок (алгоритм RMVPE) и обучается модель (150 эпох, ~7-8 мин).
4. Нейросеть Demucs разделяет песню на музыку и вокал.
5. RVC заменяет оригинальный голос Трофимова на **ваш персональный голос** с сохранением всех нот и интонаций!
6. Готовый трек проигрывается прямо в блокноте и скачивается на устройство.

## Шаг 1. Проверка GPU и установка окружения Applio (RVC v2)

In [ ]:
# Проверка наличия GPU
!nvidia-smi

# Системные пакеты
!apt-get update -qq && apt-get install -y -qq ffmpeg portaudio19-dev git curl

# Клонирование Applio
import os
if not os.path.exists("/content/Applio"):
    !git clone --depth 1 https://github.com/IAHispano/Applio.git /content/Applio

# Установка зависимостей через uv
!pip install -q uv
!cd /content/Applio && uv venv .venv --python 3.10 --seed
!cd /content/Applio && uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --python /content/Applio/.venv/bin/python
!cd /content/Applio && uv pip install -q demucs --python /content/Applio/.venv/bin/python

# Скачивание предобученных весов RVC (RMVPE, ContentVec, HiFi-GAN)
!cd /content/Applio && .venv/bin/python core.py prerequisites --models --pretraineds-hifigan
print("✓ Окружение RVC v2 (Applio) и базовые модели успешно загружены!")


## Шаг 2. Загрузка вашего чистого голоса и песни

In [ ]:
import os, shutil

# Клонируем / обновляем репозиторий с аудио
if not os.path.exists("/content/piper-voice"):
    !git clone --depth 1 https://github.com/Holki9/piper-voice.git /content/piper-voice
else:
    !cd /content/piper-voice && git pull origin main

# Подготовка папки с голосом
dataset_dir = "/content/Applio/assets/datasets/MyVoice"
shutil.rmtree(dataset_dir, ignore_errors=True)
os.makedirs(dataset_dir, exist_ok=True)
shutil.copy("/content/piper-voice/raw_audio/Testr2.wav", os.path.join(dataset_dir, "Testr2.wav"))

# Копируем песню
shutil.copy("/content/piper-voice/songs/aty_baty.mp3", "/content/aty_baty.mp3")
print("✓ Твой чистый голос (9 мин) и песня Сергей Трофимов - Аты-баты загружены!")


## Шаг 3. Предобработка и извлечение признаков (RMVPE)

In [ ]:
# 1. Нарезка и ресемплинг
!cd /content/Applio && .venv/bin/python core.py preprocess --model-name MyVoice --dataset-path assets/datasets/MyVoice --sample-rate 40000 --cpu-cores 2

# 2. Извлечение питча (RMVPE) и контент-эмбеддингов (ContentVec) на GPU
!cd /content/Applio && .venv/bin/python core.py extract --model-name MyVoice --sample-rate 40000 --cpu-cores 2 --gpu 0 --f0-method rmvpe --embedder-model contentvec

print("✓ Предобработка связок и тембра завершена!")


## Шаг 4. Обучение модели голоса (150 эпох)

In [ ]:
# Запуск быстрого обучения на GPU T4 (~7-8 минут на 150 эпох)
!cd /content/Applio && .venv/bin/python core.py train --model-name MyVoice --sample-rate 40000 --gpu 0 --batch-size 4 --save-every-epoch 25 --total-epoch 150

# Построение индекса поиска тембра
!cd /content/Applio && .venv/bin/python core.py index --model-name MyVoice --index-algorithm Auto

print("✓ Модель голоса MyVoice успешно обучена и проиндексирована!")


## Шаг 5. Спеть песню твоим голосом и свести с оригинальной музыкой

In [ ]:
import glob, os, subprocess
from IPython.display import Audio, display
from google.colab import files

print("1. Разделение трека на вокал и инструментал (Demucs)...")
!cd /content/Applio && .venv/bin/python -m demucs --two-stems=vocals /content/aty_baty.mp3 -o /content/separated

vocal_track = "/content/separated/htdemucs/aty_baty/vocals.wav"
no_vocal_track = "/content/separated/htdemucs/aty_baty/no_vocals.wav"

# Поиск путей обученной модели и индекса
pth_candidates = glob.glob("/content/Applio/assets/weights/MyVoice.pth") + glob.glob("/content/Applio/logs/MyVoice/*.pth")
index_candidates = glob.glob("/content/Applio/logs/MyVoice/*.index")

if not pth_candidates:
    raise FileNotFoundError("Не найден файл обученной модели .pth")
pth_file = pth_candidates[0]
index_file = index_candidates[0] if index_candidates else ""
print(f"Используем модель: {pth_file}")
if index_file:
    print(f"Используем индекс: {index_file}")

print("2. Перепеваем вокальную дорожку твоим голосом (RVC v2)...")
infer_cmd = [
    "/content/Applio/.venv/bin/python", "core.py", "infer",
    "--input-path", vocal_track,
    "--output-path", "/content/my_vocals.wav",
    "--pth-path", pth_file,
    "--f0-method", "rmvpe",
    "--pitch", "0",
    "--index-rate", "0.75",
    "--export-format", "WAV"
]
if index_file:
    infer_cmd.extend(["--index-path", index_file])

subprocess.run(infer_cmd, cwd="/content/Applio", check=True)

print("3. Сведение твоего вокала с оригинальной аранжировкой...")
mix_cmd = [
    "ffmpeg", "-y",
    "-i", "/content/my_vocals.wav",
    "-i", no_vocal_track,
    "-filter_complex", "[0:a]volume=1.15[v];[1:a]volume=0.95[m];[v][m]amix=inputs=2:duration=first:dropout_transition=2",
    "-b:a", "320k",
    "/content/Трофимов_Аты_баты_Мой_голос.mp3"
]
subprocess.run(mix_cmd, check=True)

print("\n🎉 ГОТОВО! Песня Сергей Трофимов — Аты-баты перепета твоим голосом!")
display(Audio("/content/Трофимов_Аты_баты_Мой_голос.mp3"))

# Скачивание готовой песни и модели на устройство
print("Скачивание готового трека...")
files.download("/content/Трофимов_Аты_баты_Мой_голос.mp3")
if pth_file and os.path.exists(pth_file):
    files.download(pth_file)
if index_file and os.path.exists(index_file):
    files.download(index_file)
